In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Carregue a planilha "Transactions". Antes de seguir, confira as primeiras linhas: o cabeçalho foi
lido corretamente? Se não, ajuste a leitura.

In [ ]:
dados = pd.read_excel('/content/drive/MyDrive/data science/KPMG_VI_New_raw_data_update_final.xlsx',sheet_name='Transactions',skiprows=1)
dados.head()

,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,list_price,standard_cost,product_first_sold_date
0,1,2,2950,2017-02-25,0.0,Approved,Solex,Standard,medium,medium,71.49,53.62,41245.0
1,2,3,3120,2017-05-21,1.0,Approved,Trek Bicycles,Standard,medium,large,2091.47,388.92,41701.0
2,3,37,402,2017-10-16,0.0,Approved,OHM Cycles,Standard,low,medium,1793.43,248.82,36361.0
3,4,88,3135,2017-08-31,0.0,Approved,Norco Bicycles,Standard,medium,medium,1198.46,381.10,36145.0
4,5,78,787,2017-10-01,1.0,Approved,Giant Bicycles,Standard,medium,large,1765.30,709.48,42226.0


2. Verifique se existem duplicatas exatas (linhas completamente iguais).

In [ ]:
print("Dimensões originais:", dados.shape)

Dimensões originais: (20000, 13)


In [ ]:
duplicatas = dados.duplicated().sum()
print("Duplicatas encontradas:", duplicatas)

Duplicatas encontradas: 0


3. Verifique duplicatas por subconjunto de colunas: quais colunas, juntas, deveriam identificar
uma transação única? Escolha e justifique.

In [ ]:
print("Duplicatas por transaction_id:", dados.duplicated(subset=['transaction_id']).sum()
)

Duplicatas por transaction_id: 0


In [ ]:
#Combinação de colunas
colunas = ['customer_id', 'product_id', 'transaction_date']

print("Duplicatas pela combinação:", dados.duplicated(subset=colunas).sum())

Duplicatas pela combinação: 1


In [ ]:
#Vizualizar duplicata
dados[
    dados.duplicated(
        subset=['customer_id', 'product_id', 'transaction_date'],
        keep=False
    )
].sort_values(['customer_id', 'product_id', 'transaction_date'])

,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,list_price,standard_cost,product_first_sold_date
10910,10911,0,1840,2017-07-11,0.0,Approved,OHM Cycles,Standard,medium,medium,235.63,125.07,36367.0
16026,16027,0,1840,2017-07-11,0.0,Approved,Solex,Standard,medium,large,202.62,151.96,33364.0


A coluna `transaction_id` é a mais adequada para identificar uma transação única, pois não apresentou duplicatas. A combinação `customer_id`, `product_id` e `transaction_date` apresentou uma repetição.

4. Investigue as colunas de identificação (como transaction_id): elas são realmente únicas?

In [ ]:
print("Quantidade de IDs únicos:", dados['transaction_id'].nunique())
print("Quantidade total de linhas:", len(dados))
print("IDs duplicados:", dados['transaction_id'].duplicated().sum())

Quantidade de IDs únicos: 20000
Quantidade total de linhas: 20000
IDs duplicados: 0


A coluna `transaction_id` identifica cada transação individualmente e não apresenta problemas de duplicidade.

5. Defina uma estratégia para os casos encontrados (remover, manter o registro mais recente,
consolidar informações...) e registre quantos registros foram afetados e qual o impacto em
pelo menos um indicador (ex.: soma de list_price).

In [ ]:
soma_antes = dados['list_price'].sum()

In [ ]:
# Remover duplicatas pela combinação escolhida, mantendo o primeiro registro
dados_limpos = dados.drop_duplicates(
    subset=['customer_id', 'product_id', 'transaction_date'],
    keep='first'
).copy()

In [ ]:
soma_depois = dados_limpos['list_price'].sum()

print("Registros antes:", len(dados))
print("Registros depois:", len(dados_limpos))
print("Registros removidos:", len(dados) - len(dados_limpos))
print("Soma de list_price antes:", round(soma_antes, 2))
print("Soma de list_price depois:", round(soma_depois, 2))
print("Diferença na soma:", round(soma_antes - soma_depois, 2))

Registros antes: 20000
Registros depois: 19999
Registros removidos: 1
Soma de list_price antes: 22156588.98
Soma de list_price depois: 22156386.36
Diferença na soma: 202.62


**Para refletir**

* Se o resultado de duplicatas exatas for zero, isso garante que não há registros repetidos? Por quê?

Não. Como vimos, podem existir registros com algumas colunas repetidas, como o mesmo cliente, produto e data, mas com outros valores diferentes.

* Em uma base de transações, duas linhas idênticas são sempre um erro? Dê um exemplo em que não seriam.

Não necessariamente. Por exemplo, duas compras diferentes podem ter os mesmos dados de produto, quantidade, valor e cliente. Se o sistema não registrar um identificador diferente para cada compra, as linhas podem parecer duplicadas, mesmo representando transações legítimas.

* Por que a escolha das colunas que definem "o mesmo registro" é uma decisão de negócio,e não apenas técnica? Relacione com a etapa de Entendimento do Negócio do CRISP-DM.

Porque é necessário entender como as transações funcionam na prática para decidir quais informações tornam cada registro único. Na etapa de Entendimento do Negócio do CRISP-DM, são definidos os objetivos e as regras do problema. Assim, evita-se excluir dados importantes por considerar duplicados registros que representam compras legítimas.

**Desafio**

In [ ]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.1 MB/s eta 0:00:00


In [ ]:
from rapidfuzz import fuzz
from itertools import combinations

In [54]:
marcas = sorted(
    dados['brand'].dropna().astype(str).str.strip().unique()
)

for marca1, marca2 in combinations(marcas, 2):
    similaridade = fuzz.ratio(marca1.lower(), marca2.lower())

    if similaridade >= 60 and marca1.lower() != marca2.lower():
        print(f'{marca1} | {marca2} | {similaridade:.1f}%')

Giant Bicycles | Norco Bicycles | 71.4%
Giant Bicycles | Trek Bicycles | 74.1%
Norco Bicycles | OHM Cycles | 66.7%
Norco Bicycles | Trek Bicycles | 74.1%
OHM Cycles | Trek Bicycles | 60.9%


In [51]:
caminho = '/content/drive/MyDrive/data science/KPMG_VI_New_raw_data_update_final.xlsx'

clientes = pd.read_excel(
    caminho,
    sheet_name='CustomerDemographic',
    header=None
)

In [52]:
clientes.columns = clientes.iloc[0]
clientes = clientes.iloc[1:].reset_index(drop=True)
clientes.columns = clientes.columns.astype(str).str.strip()

In [53]:
print('Categorias encontradas:')
print(clientes['gender'].value_counts(dropna=False))

print('\nValores diferentes:')
print(clientes['gender'].unique())

print('\nQuantidade de categorias:')
print(clientes['gender'].nunique())

Categorias encontradas:
gender
Female    2039
Male      1873
U           88
Name: count, dtype: int64

Valores diferentes:
['U' 'Male' 'Female']

Quantidade de categorias:
3
